# 项目总览：从理解数据到可验证的实验

这个 Notebook 是我的**思路索引**，不负责模型训练。具体代码、输出和证据分别保留在各阶段 Notebook；这里仅回答四个问题：

1. 数据和比赛指标到底是什么？
2. 本地分数是否足以指导实验？
3. 不做特征工程时，CatBoost 能达到什么水平？
4. 新特征是否带来了可重复的真实提升？

前三个问题已经基本解决，当前重点是第四个。

**完整路线：**

`理解数据 → 跑通 Baseline → 建立可靠 CV → 校准训练轮数 → 固定无特征工程基准 → 分阶段筛选新特征 → 尝试其他模型与融合`

当前最强的无特征工程 CatBoost：

- 5-Fold CV：**0.964007 ± 0.000473**
- Public LB：**0.96538**
- 全量训练轮数：**8,223**

这组结果是后续实验必须面对的基准，而不是可以随意更换的参考数字。

In [ ]:
import sys
import platform

import numpy
import pandas
import sklearn
import catboost
import matplotlib
import ipykernel
import jupyterlab

print("Python:", sys.version)
print("Platform:", platform.platform())

print("numpy:", numpy.__version__)
print("pandas:", pandas.__version__)
print("sklearn:", sklearn.__version__)
print("catboost:", catboost.__version__)
print("matplotlib:", matplotlib.__version__)
print("ipykernel:", ipykernel.__version__)
print("jupyterlab:", jupyterlab.__version__)

## 1. 先弄清题目、数据和评分

对应 Notebook：[01_eda.ipynb](01_eda.ipynb)

| 项目 | 结论 |
|---|---|
| Train | 691,369 行 |
| Test | 296,302 行 |
| Target | `addicted_label`，二分类 |
| 原始预测特征 | 12 个：9 个数值特征、3 个类别特征 |
| 正负样本 | 1 类约 70.9%，0 类约 29.1% |

Sample Submission 给每个测试样本都填了约 `0.709424`，它对应训练集的正类比例。这让我明确了一件基础但关键的事：**提交的是属于正类的概率，不是最终的 0/1 标签。**

### AUC 应该怎样理解

AUC 不是“预测正确率”。它衡量的是排序能力：随机取一个正样本和一个负样本，模型把正样本排在负样本前面的概率。

- `0.5`：接近随机排序
- `1.0`：排序完全正确
- 本比赛中，AUC 越高越好

### EDA 对建模的直接影响

- 所有预测特征都存在缺失，缺失率约为 4.2%–19.4%；Train/Test 的最大缺失率差约为 3.4 个百分点。缺失模式值得后续验证，但不能仅凭观察就认定它有效。
- `id` 基本是唯一标识符，第一版先移除，避免模型学习无意义的编号关系。
- 类别不均衡要求数据划分保持正负比例，因此使用分层抽样。
- CatBoost 能直接处理数值缺失和类别特征，适合作为第一个低预处理成本的基准模型。数值缺失保留为 `NaN`，类别缺失统一标记为 `"Missing"`。

EDA 的作用是提出线索和约束，不是当场把每个问题都“处理掉”。每条线索最终仍要由受控实验验证。

## 2. Baseline：先建立可工作的基准线

对应 Notebook：[02_baseline.ipynb](02_baseline.ipynb)

第一版的目标不是冲高分，而是跑通完整流程，并为后续实验留下一个可比较的起点。

| 设置 | 第一版方案 |
|---|---|
| 模型 | CatBoostClassifier |
| 训练轮数 | 500 |
| 学习率 / 深度 | 0.05 / 6 |
| 验证 | 80/20 分层 Holdout |
| 特征工程 | 无 |
| `id` | 移除 |

结果：

- Holdout AUC：**0.950917**
- 第一次提交 Public LB：**0.95227**

两个分数只差约 `0.00135`，说明本地结果与线上榜单没有明显脱节，但一次划分仍不足以证明验证体系可靠。

更重要的信号来自训练过程：最佳轮数是 **500 / 500**，验证 AUC 到最后仍在上升。这里不能把 500 理解为“最优轮数”；它只说明模型在达到上限时仍未收敛，很可能被人为截停。

因此我得到两个待验证的假设：

1. 单次 Holdout 可能受随机划分影响，需要更稳定的本地评价。
2. CatBoost 在 500 轮时明显训练不足。

Baseline 的特征重要性还提供了一条方向：`daily_screen_time_hours`、`weekend_screen_time` 和 `social_media_hours` 合计约占 76.1%。这只说明它们对当前模型的预测有用，不代表因果关系，也不代表低重要性特征永远无效。

## 3. 先把本地验证做可靠，再优化模型

对应 Notebook：[03_cv_experiments.ipynb](03_cv_experiments.ipynb)

我没有立刻增加训练轮数，而是先确认 `0.950917` 是否只是某次划分的偶然结果。为此改用分层 5-Fold Cross Validation：每次用四份训练、一份验证，轮换五次。

| 指标 | 含义 |
|---|---|
| CV Mean | 五次验证的平均水平，用来比较模型强弱 |
| CV Std | 五个 Fold 之间的波动，用来观察稳定性 |

500 轮 CatBoost 的结果：

- 5-Fold CV Mean：**0.951449**
- 5-Fold CV Std：**0.000527**
- 五折分数范围：`0.950670–0.952284`

Holdout `0.950917`、5-Fold Mean `0.951449` 和 Public LB `0.95227` 彼此接近，五折波动也很小。由此可以把当前 CV 视为可信的实验导航，而不必每改一次模型就提交 Kaggle。

需要注意：CV Std 不是模型分数，也不能机械地解释为“变化小于一个 Std 就一定无效”。实验判断还要看同折配对差值、多个 Fold 是否一致，以及改动是否可复现。

## 4. 用受控实验确认：主要瓶颈是训练不足

验证体系确定后，我保持特征、Fold、学习率、深度和随机种子不变，只调整训练预算。

| 阶段 | 设置 | 结果 | 解释 |
|---|---|---|---|
| Baseline CV | 500 轮 | `0.951449 ± 0.000527` | 最佳轮数撞上限，训练不足 |
| Exp 01 | 上限 3,000，Early Stopping | `0.962744 ± 0.000459` | 几乎所有 Fold 再次撞上限 |
| 单折 Probe | 上限 10,000，Early Stopping | Fold 1 `0.963351`，最佳轮数 8,044 | 首次找到大致收敛区间 |
| 正式 5-Fold | 上限 10,000，ESR 200 | **`0.964007 ± 0.000473`** | 最佳轮数 7,514–9,145，均值 8,223 |

这里最关键的区别是：**10,000 是允许模型寻找最佳点的上限，8,223 才是根据五折结果确定的最终训练预算。**

随后用全部 691,369 条训练数据固定训练 8,223 轮，不再保留验证集，Public LB 达到 **0.96538**。

这段实验链给出的核心结论：

- 500 → 3,000 轮带来 `+0.011295` CV AUC，是目前最大的单步提升。
- 3,000 轮之后继续寻找收敛区间，又带来 `+0.001263`。
- 相比 500 轮基准，最终 CV 共提高 `+0.012558`；Public LB 提高 `+0.01311`。
- 这些提升发生在**没有增加人工特征**的情况下，主要原因是把 CatBoost 训练充分，而不是依靠复杂技巧。

因此，在进入特征工程前，我已经先把模型本身的训练预算校准到合理区间。

## 5. Feature Engineering：只接受同协议下的稳定提升

对应 Notebook：[04_feature_engineering.ipynb](04_feature_engineering.ipynb)

进入特征工程后，最容易混淆的是三类分数：

| 数字 | 身份 | 用途 |
|---:|---|---|
| `0.963389` | 当前无特征工程 CatBoost 的 Fold 1 AUC | 低成本筛选候选特征 |
| `0.964007` | 当前无特征工程 CatBoost 的 5-Fold Mean | 本地正式基准 |
| `0.96538` | 当前 Public LB | 线上参考，不用于频繁调参 |

Fold 1 只负责快速淘汰明显无效的想法，不能代替完整 5-Fold。当前实验协议是：

1. 每个特征先写清假设，只改与该假设有关的内容。
2. `BASE` 与候选特征必须使用相同的协议、Fold、随机种子、模型设置和线程数。
3. 先比较 Fold 1 的同折配对差值；很小的正增益不自动等于有效。
4. 通过初筛后再用 Fold 2 确认，最后才运行完整 5-Fold。
5. 最终同时检查平均增益、正向 Fold 数量、波动和异常值，再决定 Keep 或 Reject。

### 5.1 第一轮：BASE 与 FE01–FE18 的 Fold 1 初筛

我先在相同 Fold 1、相同模型参数和相同 10 线程设置下，依次运行 `BASE` 与 FE01–FE18。18 个候选中有 8 个得到正 Δ，但大多数提升远小于筛选尺度；只有 FE09 和 FE10 达到进入 Fold 2 的正式门槛。

| 特征组 | Fold 1 观察 | 结论 |
|---|---|---|
| FE01–FE02：缺失信息 | 都是极小正增益，最高仅 `+0.086σ` | Reject |
| FE03–FE08：时间差与时间比例 | FE05、FE07 弱正向，其余接近零或负向 | 全部 Reject |
| FE09–FE10：活动时间构成 | FE09 `+0.780σ`；FE10 `+1.259σ` | 两者进入 Fold 2 |
| FE11–FE18：频率、睡眠与离线比例 | 多数负向；FE16 虽为 `+0.229σ`，仍未达到门槛 | 全部 Reject |

这一步没有因为合格数量少而事后放宽标准。负结果被保留下来，用来避免以后重复尝试同一批无效比例特征。

### 5.2 第二轮：FE09 与 FE10 的 Fold 2 确认

FE09 和 FE10 在新的 Fold 2 上仍然都超过同折 `BASE`：

| ID | 特征 | Fold 1 Δ | Fold 2 Δ | 结论 |
|---|---|---:|---:|---|
| FE09 | `known_activity_time` | `+0.000369` | `+0.000295` | Confirmed |
| FE10 | `unknown_activity_time` | `+0.000596` | `+0.000551` | Confirmed |

两个候选在相互独立的 Fold 上都保持正向，因此不再把 Fold 1 的结果当作偶然提升，而是将两者都推进到完整 5-Fold。

### 5.3 完整 5-Fold：FE10 明显强于 FE09

| ID | 5-Fold Mean | AUC Std | Mean Δ | 正向 Fold |
|---|---:|---:|---:|---:|
| BASE | `0.964007` | `0.000473` | `0.000000` | - |
| FE09 | `0.964348` | `0.000486` | `+0.000341` | 5/5 |
| FE10 | **`0.964629`** | `0.000523` | **`+0.000621`** | **5/5** |

FE09 和 FE10 都在五个 Fold 上稳定为正，但 FE10 的平均提升约为 FE09 的 **1.82 倍**，因此 FE10 成为当前最强的本地 CV 候选。

### 5.4 FE19：组合 FE09 与 FE10 后没有稳定胜过 FE10

看到 FE09 和 FE10 都有效后，我新增 FE19，同时加入 `known_activity_time` 与 `unknown_activity_time`，用于检查两者是否能够互补。

| Fold | FE10 | FE19 | 直接比较 |
|---|---:|---:|---|
| Fold 1 AUC | `0.963985` | `0.963972` | FE19 低 `0.000013` |
| Fold 2 AUC | `0.964452` | `0.964601` | FE19 高 `0.000149` |

FE19 并不是两个 Fold 都差于 FE10：它在 Fold 1 略低、Fold 2 略高，尚未表现出跨 Fold 一致的优势。与此同时，FE10 已有完整 5-Fold 的 5/5 正向证据，而 FE19 目前只有两个 Fold；FE19 还额外加入了可由原始活动字段求和得到的 FE09，特征集合更复杂。

因此当前主线只保留 **FE10 `unknown_activity_time`**。FE19 的准确状态是“暂不采用、尚未完成 5-Fold”，而不是“已经被完整实验否决”；如果以后重新考虑 FE19，应补跑 Fold 3–5 后再下最终结论。

## 6. 文件分工、关键数字与下一步

### 文件分工

| 文件 | 负责回答的问题 |
|---|---|
| `00_project_overview.ipynb` | 为什么做、结果意味着什么、下一步为什么这样走 |
| `01_eda.ipynb` | 数据是什么，有哪些质量问题和建模线索 |
| `02_baseline.ipynb` | 第一条完整训练与提交流程能否跑通 |
| `03_cv_experiments.ipynb` | 本地验证是否可靠，CatBoost 需要训练多久 |
| `04_feature_engineering.ipynb` | 特征假设、可比结果、筛选决策与下一步 |
| `experiments.md` | 已确认实验的简明事实记录 |
| `src/` | 特征定义、训练协议、筛选规则和结果汇总的执行来源 |

### 关键数字速查

| 数字 | 含义 |
|---:|---|
| `0.950917` | 500 轮单次 Holdout AUC |
| `0.951449 ± 0.000527` | 500 轮 5-Fold 基准 |
| `0.962744 ± 0.000459` | 3,000 轮中间结果 |
| `0.963389` | 无特征工程 CatBoost 的 Fold 1 AUC |
| `0.964007 ± 0.000473` | 无特征工程 CatBoost 的正式 5-Fold 基准 |
| `0.964348 ± 0.000486` | FE09 的完整 5-Fold 结果 |
| `0.964629 ± 0.000523` | FE10 的完整 5-Fold 结果，也是当前最佳本地 CV |
| `+0.000621` | FE10 相对同折 BASE 的平均提升，5/5 Fold 为正 |
| `8,223` | 无特征工程 v2 的五折最佳轮数均值与全量训练预算 |
| `7,561` | FE10 的五折最佳轮数均值，尚未据此完成最终全量训练 |
| `0.96538` | 当前 Public LB，来自无特征工程 v2 |

### 当前结论与下一步

1. 当前特征工程主线只保留 FE10 `unknown_activity_time`。
2. FE09 虽然 5/5 Fold 为正，但平均提升明显低于 FE10，不作为当前首选。
3. FE19 只有 Fold 1/2 证据，当前不纳入主线；若重新考虑，必须补完 Fold 3–5。
4. FE10 尚未产生 Public LB 证据；最终训练与提交时，要明确区分本地 CV 最佳和已提交模型。

到目前为止，最重要的经验不是某个模型参数，而是一条实验纪律：**先建立可信基准，再控制变量；先用便宜实验筛选，最后用完整证据定结论。**